# Rubin-like galaxy-cluster lens with PCAT

A **synthetic**, single-band image of one cluster-scale, circular isothermal lens is simulated at 0.2 arcsec per pixel with 0.7 arcsec Gaussian seeing. The background galaxy is a smooth elliptical Gaussian. PCAT samples the cluster Einstein radius and the two source coordinates from a Poisson image likelihood. This is a controlled illustration, not Rubin Observatory data or a reconstruction of a real multi-halo cluster. The foreground galaxy light, neighboring cluster members, correlated sky noise, and PSF uncertainty are omitted.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pcat
from pcat.roman_lens import RomanLensConfig, poisson_lens_log_likelihood, render_lens_counts

output_root = Path(pcat.__file__).resolve().parents[1] / 'examples/simulated_rubin_cluster_lens'
config = RomanLensConfig(
    number_side=160,
    pixel_scale=0.2,  # [arcsec pixel^-1]
    psf_fwhm=0.7,  # [arcsec]
    background=25.0,  # [electron pixel^-1]
    read_noise=0.0,  # [electron pixel^-1], for a Poisson-only mock
    source_counts=300_000.0,  # [electron]
)
source_size = 0.7  # [arcsec]
source_axis_ratio = 0.65
source_angle = 0.45  # [rad]
true_parameters = np.array([10.5, 0.7, 0.4])  # [arcsec] Einstein radius, source x, source y
random = np.random.default_rng(20260929)

expected_counts = render_lens_counts(
    config, true_parameters, source_size, source_axis_ratio, source_angle
)  # [electron pixel^-1]
observed_counts = random.poisson(expected_counts)  # [electron pixel^-1]
assert observed_counts.shape == (config.number_side, config.number_side)
print(f'Simulated {observed_counts.sum():,} detected electrons on {config.number_side} x {config.number_side} pixels.')

## Fit the image

PCAT's generic fixed-dimensional sampler evaluates a Poisson likelihood on every pixel. The source size, shape, total brightness, sky background, and seeing are held at their injected values. These short chains demonstrate the workflow; inspect mixing and run longer chains before interpreting credible intervals.

In [ ]:
run_name = 'rubin_cluster_mock'
posterior = pcat.main.sample(
    typeexpr='gener',
    retr_llik=poisson_lens_log_likelihood,
    lens_config=config,
    lens_observed_counts=observed_counts,
    lens_source_size=source_size,  # [arcsec]
    lens_source_axis_ratio=source_axis_ratio,
    lens_source_angle=source_angle,  # [rad]
    parameter_names=('einstein_radius', 'source_x', 'source_y'),
    prior_types=('self', 'self', 'self'),
    prior_minima=(8.0, -2.0, -2.0),  # [arcsec]
    prior_maxima=(13.0, 2.0, 2.0),  # [arcsec]
    initial_values=(10.0, 0.0, 0.0),  # [arcsec]
    proposal_scales=(0.08, 0.03, 0.03),  # [arcsec]
    propwithsing=True,
    probpropblock=0.0,
    pathbase=str(output_root),
    strgcnfg=run_name,
    numbproc=1,
    numbswep=400,
    numbburn=150,
    numbsamp=250,
    booladaptstdp=True,
    typeseed=42,
    typeverb=-1,
)
draws = np.asarray(posterior.listpostparagenrscalbase)
assert draws.shape[1] == 3 and np.all(np.isfinite(draws))
median_parameters = np.median(draws, axis=0)  # [arcsec]
print(f'Posterior median Einstein radius: {median_parameters[0]:.2f} arcsec')

In [ ]:
from io import BytesIO
from IPython.display import Image, display

typefileplot = 'png'  # choose 'png' or 'pdf'
assert typefileplot in ('png', 'pdf')
visual_root = output_root / 'visuals'
print(f'Writing to {visual_root}...')
visual_root.mkdir(parents=True, exist_ok=True)
model_counts = render_lens_counts(
    config, median_parameters, source_size, source_axis_ratio, source_angle
)  # [electron pixel^-1]
standard_residual = (observed_counts - model_counts) / np.sqrt(model_counts)
field_half_width = config.number_side * config.pixel_scale / 2.0  # [arcsec]
extent = (-field_half_width, field_half_width, -field_half_width, field_half_width)
count_limit = np.percentile(observed_counts, 99.5)  # [electron pixel^-1]
figure, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
for axis, image, title, bounds, colorbar_label in zip(
    axes,
    (observed_counts, model_counts, standard_residual),
    ('Simulated observation', 'PCAT median model', 'Poisson residual'),
    ((0, count_limit), (0, count_limit), (-5, 5)),
    ('Electrons [pixel$^{-1}$]', 'Electrons [pixel$^{-1}$]', 'Residual [$\\sigma$]'),
):
    displayed = axis.imshow(image, origin='lower', extent=extent,
                            cmap='Greys' if title != 'Poisson residual' else 'RdBu_r',
                            vmin=bounds[0], vmax=bounds[1], interpolation='nearest')
    axis.set_title(title)
    axis.set_xlabel('Offset [arcsec]')
    axis.set_ylabel('Offset [arcsec]')
    axis.grid(False)
    figure.colorbar(displayed, ax=axis, label=colorbar_label, shrink=0.8)
image_path = visual_root / f'rubin_cluster_image_fit.{typefileplot}'
print(f'Writing to {image_path}...')
figure.savefig(image_path, dpi=300, bbox_inches='tight')
preview = BytesIO()
figure.savefig(preview, format='png', dpi=150, bbox_inches='tight')
display(Image(data=preview.getvalue()))
plt.close(figure)

## Parameter recovery

The curves below summarize this finite PCAT chain, with the injected values shown for comparison. They are illustrative posterior marginals, not validated cluster constraints.

In [ ]:
parameter_labels = ('Einstein radius [arcsec]', 'Source x [arcsec]', 'Source y [arcsec]')
figure, axes = plt.subplots(1, 3, figsize=(11, 3.3), constrained_layout=True)
for index, (axis, label) in enumerate(zip(axes, parameter_labels)):
    axis.hist(draws[:, index], bins=22, density=True, color='#007360', alpha=0.75,
              label='PCAT samples')
    axis.axvline(true_parameters[index], color='#A51417', linewidth=2,
                 label='Injected value')
    axis.axvline(median_parameters[index], color='black', linestyle='--', linewidth=1.5,
                 label='Posterior median')
    axis.set_xlabel(label)
    axis.set_ylabel('Density [arcsec$^{-1}$]')
    axis.grid(False)
axes[0].legend(loc='upper left', frameon=True, fancybox=True, framealpha=1.0, fontsize=8)
posterior_path = visual_root / f'rubin_cluster_posterior.{typefileplot}'
print(f'Writing to {posterior_path}...')
figure.savefig(posterior_path, dpi=300, bbox_inches='tight')
preview = BytesIO()
figure.savefig(preview, format='png', dpi=150, bbox_inches='tight')
display(Image(data=preview.getvalue()))
plt.close(figure)
for label, values in zip(parameter_labels, draws.T):
    lower, middle, upper = np.percentile(values, (16, 50, 84))  # [arcsec]
    print(f'{label}: {middle:.2f} [{lower:.2f}, {upper:.2f}] (16th, 84th percentiles)')